# Loyal Servant Role

Loyal Servant is on the Good team and doesn't know anything
Loyal Servant's job is to **help Good pass 3 missions** while **not letting Merlin be identified**.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

In [2]:
class MissionResult(str, Enum):
    SUCCESS = "success"
    FAIL = "fail"

# constrain the actions
class VoteDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"


class MissionAction(str, Enum):
    PASS = "pass"
    FAIL = "fail"


@dataclass
class Proposal:
    round_idx: int
    proposal_idx: int
    proposer: str
    team: List[str]
    votes: Dict[str, VoteDecision] = field(default_factory=dict)
    approved: Optional[bool] = None


@dataclass
class Mission:
    round_idx: int
    team: List[str]
    outcome: Optional[MissionResult] = None
    fail_count: Optional[int] = None

# the environment 
# gives the information at decision time
@dataclass
class GameState:
    players: List[str]
    me: str
    round_idx: int
    proposal_idx: int
    num_successes: int
    num_fails: int
    current_proposer: str
    current_team: List[str]
    chat_log: List[Tuple[str, str]] = field(default_factory=list)
    proposals: List[Proposal] = field(default_factory=list)
    missions: List[Mission] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)

In [ ]:
# we just plug in our stuff and then it generates
class LLMBackend(Protocol):
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        ...


class FakeLLM:
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        import random
        import json

        lower = user.lower()

        if '"type":"vote"' in lower:
            # Neutral Good voting behavior
            decision = "approve" if random.random() < 0.6 else "reject"
            return json.dumps({
                "type": "vote",
                "decision": decision,
                "reason": "Based on current discussion and voting patterns."
            })

        if '"type":"mission"' in lower:
            # Loyal Servant (Good) always passes missions.
            return json.dumps({
                "type": "mission",
                "action": "pass",
                "reason": "Supporting mission success."
            })

        return json.dumps({
            "type": "speech",
            "message": "I'm trying to track patterns in votes and team compositions. Let's stay consistent.",
            "intent": "discussion"
        })

In [ ]:
# basically an edge case checker
# will try to fix if the LLM tries to respond in json
def _extract_json_object(text: str) -> Optional[dict]:
    text = text.strip()
    if not text:
        return None

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def require_fields(obj: dict, fields: List[str]) -> bool:
    return all(f in obj for f in fields)

In [ ]:
# main prompt
def build_loyal_servant_system_prompt(me: str) -> str:
    return (
        "You are playing The Resistance: Avalon.\n"
        "Your role: LOYAL SERVANT OF ARTHUR (GOOD).\n\n"
        f"You are: {me}\n\n"
        "What you know:\n"
        "- You do NOT know who Evil is.\n\n"
        "Goals:\n"
        "1) Help Good complete 3 successful missions.\n"
        "2) Use public information (votes, mission outcomes, chat) to form suspicions.\n"
        "3) Communicate like a normal player (no perfect knowledge).\n\n"
        "Style constraints (very important):\n"
        "- Use hedged language (e.g., 'I feel', 'maybe', 'my read is...').\n"
        "- Keep reasons short (1-2 sentences).\n"
        "- Never claim certainty about someone's alignment.\n\n"
        "When asked for actions, respond ONLY in valid JSON matching the requested schema."
    )


In [ ]:
# A Loyal Servant does not know Evil.
# We track lightweight suspicion scores inferred from PUBLIC information.
@dataclass
class LoyalServantMemory:
    # suspicion[p] higher => more suspicious
    suspicion: Dict[str, float] = field(default_factory=dict)

    def bump(self, player: str, delta: float) -> None:
        self.suspicion[player] = self.suspicion.get(player, 0.0) + delta

    def decay_all(self, factor: float = 0.95) -> None:
        for p in list(self.suspicion.keys()):
            self.suspicion[p] *= factor


In [ ]:
# things it can do
class LoyalServantAgent:
    def __init__(self, *, llm: LLMBackend, me: str):
        self.llm = llm
        self.me = me
        self.system = build_loyal_servant_system_prompt(me)
        self.memory = LoyalServantMemory()

    def _update_suspicion_from_history(self, state: GameState) -> None:
        """Update suspicion using mission outcomes (public info).

        Heuristic:
        - On a FAILED mission: everyone on that team becomes more suspicious.
        - On a SUCCESS mission: everyone on that team becomes slightly less suspicious.
        - Apply mild decay so old evidence matters less.
        """
        self.memory.decay_all(0.97)

        for m in state.missions:
            if m.outcome is None:
                continue
            if m.outcome == MissionResult.FAIL:
                for p in m.team:
                    if p != self.me:
                        self.memory.bump(p, 0.35)
            elif m.outcome == MissionResult.SUCCESS:
                for p in m.team:
                    if p != self.me:
                        self.memory.bump(p, -0.15)

    def speak(self, state: GameState) -> str:
        self._update_suspicion_from_history(state)

        # Build a small, human-looking summary for the model
        top_sus = sorted(
            ((p, s) for p, s in self.memory.suspicion.items() if p != self.me),
            key=lambda x: x[1],
            reverse=True
        )[:2]
        sus_text = ", ".join([f"{p}({s:+.2f})" for p, s in top_sus]) if top_sus else "none yet"

        user = (
            '{"type":"speech","message":string,"intent":string}\n\n'
            + build_state_block(state)
            + f"\nMy current (soft) suspicions: {sus_text}\n"
            + "Task: Produce 1-3 sentences for discussion. Be helpful and hedged."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        msg = obj.get("message", "").strip()
        if msg:
            return msg
        return "I'm not fully sold yet — can we talk through why this team makes sense?"

    def vote_on_team(self, state: GameState, team: List[str]) -> VoteDecision:
        self._update_suspicion_from_history(state)

        # Basic heuristic: reject if team contains someone very suspicious,
        # but don't be perfectly consistent.
        sus_scores = [self.memory.suspicion.get(p, 0.0) for p in team if p != self.me]
        max_sus = max(sus_scores) if sus_scores else 0.0

        user = (
            '{"type":"vote","decision":"approve|reject","reason":string}\n\n'
            + build_state_block(state)
            + f"\nVote on team: {team}\n"
            + "Task: Choose approve/reject and give a short, hedged reason."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        decision = obj.get("decision", "").lower()

        # Guardrails based on suspicion (public, fallible logic)
        if max_sus >= 0.6:
            return VoteDecision.REJECT
        if max_sus <= 0.1:
            return VoteDecision.APPROVE

        # Middle zone: let the model's choice decide, default approve
        return VoteDecision.REJECT if decision == "reject" else VoteDecision.APPROVE

    def mission_action(self, state: GameState) -> MissionAction:
        # Loyal Servant is Good and should never fail missions intentionally.
        return MissionAction.PASS

    def propose_team(self, state: GameState, team_size: int) -> List[str]:
        self._update_suspicion_from_history(state)

        # Include self most of the time (common Good play)
        team = []
        if team_size > 0 and random.random() < 0.85:
            team.append(self.me)

        # Prefer low-suspicion players (publicly inferred)
        others = [p for p in state.players if p != self.me]
        others_sorted = sorted(others, key=lambda p: self.memory.suspicion.get(p, 0.0))
        for p in others_sorted:
            if len(team) >= team_size:
                break
            if p not in team:
                team.append(p)

        # Fallback: if self wasn't included or we somehow fell short, fill deterministically
        if len(team) < team_size:
            for p in state.players:
                if p not in team and len(team) < team_size:
                    team.append(p)

        return team[:team_size]


In [ ]:
# demo
llm = FakeLLM()

agent = LoyalServantAgent(
    llm=llm,
    me="Player_A",
)

state = GameState(
    players=["Player_A", "Player_B", "Player_C", "Player_D", "Player_E"],
    me="Player_A",
    round_idx=1,
    proposal_idx=1,
    num_successes=0,
    num_fails=0,
    current_proposer="Player_A",
    current_team=["Player_A", "Player_C"],
    chat_log=[
        ("Player_B", "I think A and C seems fine."),
        ("Player_D", "Not sure, I want to see B on a team first.")
    ],
    missions=[
        Mission(round_idx=0, team=["Player_B","Player_C"], outcome=MissionResult.SUCCESS, fail_count=0),
        Mission(round_idx=0, team=["Player_B","Player_D"], outcome=MissionResult.FAIL, fail_count=1),
    ],
)

print("Loyal Servant says:", agent.speak(state))
print("Loyal Servant votes:", agent.vote_on_team(state, state.current_team).value)
print("Loyal Servant mission action:", agent.mission_action(state).value)
print("Loyal Servant proposed team (size 3):", agent.propose_team(state, 3))


Speech: I’m not convinced yet. Let’s evaluate voting consistency.
Vote: VoteDecision.REJECT
Mission action: MissionAction.PASS
Assassinate: Player_D
